# MCC DAQ Examples
This notebook demonstrates the console examples from the `mcculw` library.
Requires the MCC Universal Library (InstaCal) to be installed and a DAQ device to be connected.

In [1]:
# Install the mcculw package from this repo (only needs to run once)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])

0

In [3]:
import sys, os

# Make the console examples importable
sys.path.insert(0, os.path.join(os.getcwd(), 'examples', 'console'))

from mcculw import ul
from mcculw.device_info import DaqDeviceInfo
from mcculw.enums import DigitalIODirection, TempScale, CounterChannelType
from console_examples_util import config_first_detected_device

print('mcculw imported successfully')

mcculw imported successfully


## Device Detection
Detect all connected DAQ devices and configure the first one as board 0.

In [4]:
board_num = 0

config_first_detected_device(board_num)

daq_dev_info = DaqDeviceInfo(board_num)
print('Active DAQ device:', daq_dev_info.product_name, '(', daq_dev_info.unique_id, ')')
print('  Supports analog input: ', daq_dev_info.supports_analog_input)
print('  Supports analog output:', daq_dev_info.supports_analog_output)
print('  Supports digital I/O:  ', daq_dev_info.supports_digital_io)

Found 1 DAQ device(s):
  USB-1808X (21A3139) - Device ID = 318
Active DAQ device: USB-1808X ( 21A3139 )
  Supports analog input:  True
  Supports analog output: True
  Supports digital I/O:   True


## Analog Input
Reads a single value from analog input channel 0 and converts it to engineering units (volts).

In [5]:
try:
    if not daq_dev_info.supports_analog_input:
        raise Exception('Device does not support analog input')

    ai_info = daq_dev_info.get_ai_info()
    ai_range = ai_info.supported_ranges[0]
    channel = 0

    if ai_info.resolution <= 16:
        value = ul.a_in(board_num, channel, ai_range)
        eng_units_value = ul.to_eng_units(board_num, ai_range, value)
    else:
        value = ul.a_in_32(board_num, channel, ai_range)
        eng_units_value = ul.to_eng_units_32(board_num, ai_range, value)

    print(f'Channel {channel}  Raw: {value}  Engineering value: {eng_units_value:.4f} V')
except Exception as e:
    print('Error:', e)

Channel 0  Raw: 142457  Engineering value: 0.8686 V


## Digital Input
Reads the value of the first available digital input port and its first bit.

In [6]:
try:
    if not daq_dev_info.supports_digital_io:
        raise Exception('Device does not support digital I/O')

    dio_info = daq_dev_info.get_dio_info()
    port = next((p for p in dio_info.port_info if p.supports_input), None)
    if not port:
        raise Exception('Device has no digital input port')

    if port.is_port_configurable:
        ul.d_config_port(board_num, port.type, DigitalIODirection.IN)

    port_value = ul.d_in(board_num, port.type)
    bit_value = ul.d_bit_in(board_num, port.type, 0)

    print(f'Port {port.type.name}: {port_value:#010b} ({port_value})')
    print(f'Bit 0: {bit_value}')
except Exception as e:
    print('Error:', e)

Port AUXPORT: 0b00000000 (0)
Bit 0: 0


## Digital Output
Writes a value to the first available digital output port.

In [7]:
try:
    if not daq_dev_info.supports_digital_io:
        raise Exception('Device does not support digital I/O')

    dio_info = daq_dev_info.get_dio_info()
    port = next((p for p in dio_info.port_info if p.supports_output), None)
    if not port:
        raise Exception('Device has no digital output port')

    if port.is_port_configurable:
        ul.d_config_port(board_num, port.type, DigitalIODirection.OUT)

    output_value = 0xFF  # All bits high — change as needed
    ul.d_out(board_num, port.type, output_value)
    print(f'Wrote {output_value:#010b} ({output_value}) to port {port.type.name}')
except Exception as e:
    print('Error:', e)

Wrote 0b11111111 (255) to port AUXPORT


## Temperature Input
Reads a temperature from channel 0 (requires a thermocouple device or EXP board).

In [8]:
try:
    if not daq_dev_info.supports_analog_input:
        raise Exception('Device does not support analog input')

    ai_info = daq_dev_info.get_ai_info()
    if ai_info.num_temp_chans <= 0:
        raise Exception('Device does not support temperature input')

    channel = 0
    value = ul.t_in(board_num, channel, TempScale.CELSIUS)
    print(f'Channel {channel}: {value:.2f} °C')
except Exception as e:
    print('Error:', e)

Error: Device does not support temperature input


## Counter Input
Reads the current count from the first available counter channel.

In [9]:
try:
    ctr_info = daq_dev_info.get_ctr_info()
    chan = next((c for c in ctr_info.chan_info
                 if c.type == CounterChannelType.CTRQUAD or
                    c.type == CounterChannelType.CTRSIMPLE), None)
    if not chan:
        raise Exception('No compatible counter channel found')

    counter_num = chan.channel_num
    ul.c_clear(board_num, counter_num)
    count = ul.c_in_32(board_num, counter_num)
    print(f'Counter {counter_num}: {count}')
except Exception as e:
    print('Error:', e)

Error: type object 'CounterChannelType' has no attribute 'CTRSIMPLE'


## Release Device
Always release the device when done.

In [10]:
ul.release_daq_device(board_num)
print('Device released')

Device released
